# 03 · Swin ReID on the frozen detector: single-pass and two-pass

Trains the ReID part of the two Swin systems of `main.ipynb`. The detector from `02_swin_detector` (`swin_detector.pt`) stays **frozen**, so only ReID parameters are in the optimizer. Run the notebook once per `STRATEGY`:

| `STRATEGY` | System in `main.ipynb` | What is trained | Gallery embedding | Output | Run time (T4) |
|---|---|---|---|---|---|
| `"crop_to_scene"` | Swin single-pass (diagram 2) | a fresh RoI ReID head on the frozen FPN features | RoIAlign on the detector's own features (one backbone pass) | `swin_single_pass.pt` | ≈ 5.3 h |
| `"crop_to_crop"` | Swin two-pass (diagram 3) | an independent ImageNet Swin-T crop encoder (last two stages + projection) | crop the original pixels, re-encode at 256×128 | `swin_two_pass.pt` | ≈ 2.9 h |

Both strategies use supervised NT-Xent (SupCon, τ = 0.1) with positives from different frames, random P×K = 16×4 identity batches, 256×128 query crops and 50 epochs. Each epoch is validated by full retrieval on the validation frames. The published files are the **latest (epoch 50)** checkpoints. `best.pt`, selected by validation mAP, is also saved.

### How to run
1. Create a Kaggle notebook from this file (*File → Import Notebook*), then attach the inputs listed below with *Add Input*.
2. Settings: **Accelerator → GPU T4 ×1**. Internet is only needed if you set `USE_WANDB = True`.
3. **Run All.** The inputs are found automatically wherever Kaggle mounts them under `/kaggle/input`.

Weights & Biases logging is **off** by default (`USE_WANDB = False`), so no account or secret is needed. To log curves and upload checkpoints, set `USE_WANDB = True` and store your key as the Kaggle secret / environment variable `WANDB_API_KEY`.

**Inputs**

| Input | Kaggle source | Used for |
|---|---|---|
| PRW dataset | `edoardomerli/prw-person-re-identification-in-the-wild` | frames, annotations, queries |
| Code | `saadqasimabbasi/prw-person-search-code` | the project modules in `src/` (and the DINOv3 code snapshot) |
| Detector | `saadqasimabbasi/prw-person-search-weights` or the output of `02_swin_detector` | `swin_detector.pt` (frozen) |

## 1. Setup: locate the Kaggle inputs
This cell scans `/kaggle/input` once and finds the project code (the `prw-person-search-code` dataset), the PRW dataset and the checkpoint files this notebook needs. It fails with a clear message if one is missing, then adds the code folder to `sys.path`. Outputs go to `/kaggle/working`. Set the environment variables `PRW_INPUT_ROOT` / `PRW_WORKING` to run outside Kaggle.

> **Run time:** a few seconds.

In [ ]:
import os
import sys
from pathlib import Path

import torch

INPUT_ROOT = Path(os.environ.get("PRW_INPUT_ROOT", "/kaggle/input"))
WORKING = Path(os.environ.get("PRW_WORKING", "/kaggle/working"))
USE_WANDB = False  # True: log curves and upload checkpoints to W&B (needs WANDB_API_KEY)
os.environ["WANDB_MODE"] = "online" if USE_WANDB else "disabled"
os.environ["WANDB_DIR"] = str(WORKING)

PROJECT_MODULES = {"model.py", "training.py", "prw_dataset.py", "prw_evaluation.py"}
REQUIRED_FILES = {"swin_detector.pt"}
SKIPPED_FOLDERS = {"frames", "annotations", "query_box", ".git"}  # thousands of images, nothing to find


def scan_inputs():
    """Walk the input folder once and return the code, DINOv3, PRW and checkpoint locations."""
    code_dir = dinov3_dir = prw_root = None
    found = {}
    for directory, subfolders, files in os.walk(INPUT_ROOT, followlinks=True):
        directory = Path(directory)
        if prw_root is None and "query_info.txt" in files and "frames" in subfolders:
            prw_root = directory
        subfolders[:] = [name for name in subfolders if name not in SKIPPED_FOLDERS]
        if code_dir is None and PROJECT_MODULES.issubset(files):
            code_dir = directory
        if dinov3_dir is None and "hubconf.py" in files and directory.name.startswith("dinov3"):
            dinov3_dir = directory
        for name in REQUIRED_FILES.intersection(files):
            found.setdefault(name, directory / name)
    return code_dir, dinov3_dir, prw_root, found


CODE_DIR, DINOV3_DIR, PRW_ROOT, INPUT_FILES = scan_inputs()
missing = [label for label, value in [("code dataset (project .py files)", CODE_DIR),
                                      ("PRW dataset", PRW_ROOT)] if value is None]
missing += [f"input file {name}" for name in sorted(REQUIRED_FILES - set(INPUT_FILES))]
if missing:
    raise FileNotFoundError("Missing inputs (see 'How to run'): " + ", ".join(missing))
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
WORKING.mkdir(parents=True, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Code:  ", CODE_DIR)
print("PRW:   ", PRW_ROOT)
for name, path in sorted(INPUT_FILES.items()):
    print("Input: ", name, "->", path)
print("Device:", device, torch.cuda.get_device_name(0) if device.type == "cuda" else "(no GPU: enable one in the settings)")

## 2. Configuration, frozen detector and model
The detector is rebuilt from `swin_detector.pt` with `detection_only=True` and loaded strictly. Its resolution (512×768) and split seed are reused.
- `crop_to_scene` wraps the detector in `SwinSceneReID` (`src/swin_reid.py`). Query crops and gallery boxes are both pooled from the frozen FPN.
- `crop_to_crop` pairs the frozen detector with a separate `SwinCropReID` encoder (`DinoPersonSearch` is the generic detector + crop-encoder wrapper).

The assertion checks that no detector parameter is trainable.

> **Run time (Kaggle T4):** ≈ 3 min (estimate; includes caching the validation detections).

In [ ]:
import json
import random

from torch.utils.data import DataLoader

from dino_reid import DinoPersonSearch
from model import PersonSearchModel
from prw_dataset import PRWPersonSearchDataset, PRWSubset, PKIdentityBatchSampler, prw_collate_fn
from swin_reid import SwinCropReID, SwinSceneReID, swin_reid_epoch
from training import make_grad_scaler, start_run
from validation_retrieval import build_validation_gallery, evaluate_validation_gallery

STRATEGY = "crop_to_scene"  # or "crop_to_crop"; run once per strategy
OUTPUT_NAME = {"crop_to_scene": "swin_single_pass.pt", "crop_to_crop": "swin_two_pass.pt"}[STRATEGY]
P, K = 16, 4
NUM_EPOCHS = 50
SCORE_THRESHOLD = .5
TEMPERATURE = .1
CROP_SIZE = (256, 128)
NUM_WORKERS = 2

detector_path = INPUT_FILES["swin_detector.pt"]
checkpoint = torch.load(detector_path, map_location="cpu", weights_only=False)
detector_config = dict(checkpoint["model_config"])
if detector_config["backbone_name"] != "swin_t":
    raise ValueError("Expected the Swin-T detector")
detector_config.update(pretrained=False, detection_only=True)
IMAGE_SIZE = tuple(checkpoint.get("detector_image_size", checkpoint["run_config"]["image_size"]))
SEED = int(checkpoint["run_config"]["seed"])
VAL_FRACTION = float(checkpoint["run_config"].get("val_fraction", .1))
random.seed(SEED)
torch.manual_seed(SEED)
detector = PersonSearchModel(**detector_config)
detector.load_state_dict(checkpoint["model_state_dict"], strict=True)
detector_epoch = checkpoint["epoch"]
del checkpoint

if STRATEGY == "crop_to_scene":
    model = SwinSceneReID(detector, crop_size=CROP_SIZE, feature_batch_size=2)
else:
    model = DinoPersonSearch(detector, SwinCropReID(pretrained=True, crop_size=CROP_SIZE))
model = model.to(device)
GALLERY_MODE = "scene" if STRATEGY == "crop_to_scene" else "crop"

full = PRWPersonSearchDataset(PRW_ROOT, split="train", image_size=IMAGE_SIZE, query_crop_size=CROP_SIZE)
indices = torch.randperm(len(full), generator=torch.Generator().manual_seed(SEED)).tolist()
cut = round(len(indices) * VAL_FRACTION)
train_data, val_data = PRWSubset(full, indices[cut:]), PRWSubset(full, indices[:cut])
train_loader = DataLoader(train_data, batch_sampler=PKIdentityBatchSampler(train_data, p=P, k=K),
                          collate_fn=prw_collate_fn, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_data, batch_sampler=PKIdentityBatchSampler(val_data, p=4, k=2, seed=SEED),
                        collate_fn=prw_collate_fn, num_workers=NUM_WORKERS)

# Only ReID parameters are optimised; the crop encoder backbone gets a lower learning rate.
if STRATEGY == "crop_to_scene":
    groups = [{"params": model.reid_head.parameters(), "lr": 1e-4}]
else:
    groups = [{"params": [p for n, p in model.reid_head.named_parameters()
                          if p.requires_grad and not n.startswith("embedding_proj.")], "lr": 1e-5},
              {"params": model.reid_head.embedding_proj.parameters(), "lr": 1e-4}]
optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
scaler = make_grad_scaler(device)
assert not any(p.requires_grad for p in model.detector.parameters()), "the detector must stay frozen"

config = dict(strategy=STRATEGY, seed=SEED, val_fraction=VAL_FRACTION, P=P, K=K, epochs=NUM_EPOCHS,
              crop_size=CROP_SIZE, image_size=IMAGE_SIZE, temperature=TEMPERATURE,
              detector_path=str(detector_path), detector_epoch=detector_epoch,
              score_threshold=SCORE_THRESHOLD, checkpoint_selection="validation_retrieval_mAP")
run = start_run(config, name=f"swin-reid-{STRATEGY}-p16-k4")
cache = build_validation_gallery(model, PRW_ROOT, [full.image_names[i] for i in val_data.indices], device,
                                 score_threshold=SCORE_THRESHOLD, detector_image_size=IMAGE_SIZE)
print(STRATEGY, "| train/validation frames:", len(train_data), len(val_data))

## 3. Train and validate
Each epoch trains on the P×K batches with `swin_reid_epoch`. It then computes the validation loss and the validation retrieval mAP over the cached validation detections. `latest.pt` is written every epoch and `best.pt` whenever validation mAP improves. At the end, `latest.pt` (epoch 50, the checkpoint published for `main.ipynb`) is copied to `/kaggle/working/<OUTPUT_NAME>`.

> **Run time (Kaggle T4):** ≈ 6 min per epoch for `crop_to_scene`, ≈ 3.5 min for `crop_to_crop`.

In [ ]:
import shutil

import wandb

checkpoint_dir = WORKING / "checkpoints" / f"swin_reid_{STRATEGY}"
checkpoint_dir.mkdir(parents=True, exist_ok=True)
best = -1.
for epoch in range(1, NUM_EPOCHS + 1):
    losses = swin_reid_epoch(model, train_loader, device, STRATEGY, optimizer, scaler, TEMPERATURE)
    validation_loss = swin_reid_epoch(model, val_loader, device, STRATEGY, temperature=TEMPERATURE)
    metrics = evaluate_validation_gallery(model, PRW_ROOT, cache, device, gallery_embedding_mode=GALLERY_MODE)
    run.log({"epoch": epoch, **{f"train/{k}": v for k, v in losses.items()},
             **{f"val/{k}": v for k, v in {**validation_loss, **metrics}.items()}})
    improved = metrics["retrieval_mAP"] > best
    best = max(best, metrics["retrieval_mAP"])
    state = dict(epoch=epoch, strategy=STRATEGY, run_config=config, detector_config=detector_config,
                 model_state_dict=model.state_dict(), optimizer_state_dict=optimizer.state_dict(),
                 scaler_state_dict=scaler.state_dict(), best_retrieval_map=best, validation_metrics=metrics)
    torch.save(state, checkpoint_dir / "latest.pt")
    if improved:
        torch.save(state, checkpoint_dir / "best.pt")
    print(f"epoch {epoch}: {metrics}; best mAP={best:.4f}", flush=True)

shutil.copy2(checkpoint_dir / "latest.pt", WORKING / OUTPUT_NAME)
if USE_WANDB:
    artifact = wandb.Artifact(f"swin-reid-{STRATEGY}-latest", type="model", metadata=config)
    artifact.add_file(str(WORKING / OUTPUT_NAME))
    run.log_artifact(artifact)
run.finish()
print("saved", WORKING / OUTPUT_NAME)

## 4. Test evaluation with the official `eval_search_prw`
The saved model is evaluated once on the official test split with detected gallery boxes, using the same gallery path as in `main.ipynb`.

> **Run time (Kaggle T4):** ≈ 15 min.

In [ ]:
from prw_evaluation import evaluate_prw

model.eval()
result = evaluate_prw(model, PRW_ROOT, device, score_threshold=SCORE_THRESHOLD, iou_threshold=.5,
                      detector_image_size=IMAGE_SIZE, gallery_embedding_mode=GALLERY_MODE, official_eval=True,
                      batch_size=2, query_batch_size=32, gallery_crop_batch_size=32)
print(f"{STRATEGY}: official mAP = {result['official_mAP']:.2%}, top-1 = {result['official_top1']:.2%}")